# Chapter 5 · Brownian Bridges for ABPs — Sampling Trajectories

Section 5.7 (pp. 104–107), Figures 41 and 42.

| Figure | Page | Content |
|---|---|---|
| Figure 41 | p. 106 | Sticky chains when sampling ABP trajectories |
| Figure 42 | p. 107 | MCMC warm-up |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import lambert_thesis as lt
from lambert_thesis import abp_inference as abp, bridges, filters, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

## 5.7 Sampling trajectories

Before the inference task we look at sampling trajectories given observations (state estimation): the unobserved
variables are the orientation and the entire thermal noise along the trajectory. We use the Correlated Pseudo-Marginal
Metropolis–Hastings algorithm (CPMMH, Wiqvist et al.): the pseudo-random numbers of the Euler–Maruyama integration become
auxiliary parameters with a standard-normal prior, proposed with a Crank–Nicolson step
$w' = \rho\,w+\sqrt{1-\rho^2}\,\xi$ whose correlation $\rho$ controls the step size.

**Setup.** Ground truth: $v=3$, $D_r=2$, observations every $\Delta T=0.1$ with measurement noise $10^{-3}$, integrated with
$\Delta t=10^{-3}$ (with null-bridge noise). The sampler uses a particle filter with 100 particles and $\Delta t=10^{-2}$ on
the first 4 observations; every chain runs 3000 iterations; the bridge conditions start from a probe trajectory (a
sample of the unbridged particle filter with 2000 particles); and every accepted sample is drawn (colour: iteration, from blue
to green).

*Code:* `abp_inference.propagate` (the ABP propagator, optionally with bridges), `filters.particle_filter`, and the noise
blocks of `bridges.CorrelatedRNG`, whose `reroll(rho)` is the Crank–Nicolson proposal.

In [ ]:
GT = dict(logv=np.log(3.0), logDr=np.log(2.0), dt=1e-3, bridge=bridges.ConditionedBridge())
T, gx, gy = abp.simulate(30, 0.1, GT, rng_seed=9)          # example trajectory (seed 9)
SIG_OBS = 1e-3
ox = gx + rng.normal(scale=SIG_OBS, size=gx.shape)
oy = gy + rng.normal(scale=SIG_OBS, size=gy.shape)
V, DR, DT = 3.0, 2.0, 1e-2
N_STEPS = int(0.1 / DT + 1)
DT_ = 0.1 / N_STEPS
SLOPE_FACTOR = 1 / np.sqrt(2 * DR * DT_)            # bridge condition (noise units) per radian of final angle
SHIFT_FACTOR = 1 / (np.sqrt(2 * DR * DT_) * V * DT_)  # ... per unit of paraxial deviation

def filter_params(use_bridges, **kw):
    return dict(logv=np.log(V), logDr=np.log(DR), dt=DT, rng=bridges.CorrelatedRNG(lt.SEED),
                bridge=bridges.ConditionedBridge(), use_bridges=use_bridges, bridge_prior="independent", **kw)

def run_filter(obs, p, n_parts, **kw):
    def before(k, X, o):
        p["target_shift"], p["target_slope"] = o.get("target_shift", 0.0), o.get("target_slope", 0.0)
    return filters.particle_filter(lambda X, o: abp.log_obs_density(X, o, SIG_OBS),
                                   lambda T0, T1, X, k: abp.propagate(T0, T1, X, k, p),
                                   np.zeros((n_parts, 3)), obs, rng, before_step=before, **kw)

def initial_observations(n_obs, probe=True, n_probe=2000):
    """Observations with bridge conditions (shift, slope) per interval, initialised from a probe trajectory:
    a posterior sample of the unbridged particle filter, whose noise in each interval has a well-defined
    endpoint B2_N and last step B1_N. Without probe, all conditions start at zero."""
    obs = abp.make_observations(T, ox, oy)[:n_obs]
    for o in obs:
        o["target_shift"] = o["target_slope"] = 0.0
    if probe:
        res = run_filter(obs, filter_params(False, record_bridge_parameters=True), n_probe, keep_history=True)
        for o, (B1, B2) in zip(obs, res["streams"]["bridge"].T):
            o["target_slope"], o["target_shift"] = B1 / SLOPE_FACTOR, B2 / SHIFT_FACTOR
    return obs

def chain(obs, rho, use_bridges, count=3000, n_parts=100, step=5e-4):
    """CPMMH chain over the noise (rho: correlation of the Crank-Nicolson proposal). With bridges, the bridge
    conditions (shift and slope of every interval) are sampled as well, with Gaussian random-walk steps."""
    p = filter_params(use_bridges, hires=True)
    logl, accepted = -np.inf, []
    for it in range(count):
        p["rng"].reroll(rho)
        if use_bridges:
            old = [(o["target_shift"], o["target_slope"]) for o in obs]
            for o in obs:
                o["target_shift"] += rng.standard_normal() * step
                o["target_slope"] += rng.standard_normal() * step
        res = run_filter(obs, p, n_parts, keep_history=True, add_propagator_logl=use_bridges)
        if rng.random() < np.exp(min(res["logl"] - logl, 0.0)):
            logl = res["logl"]
            accepted.append((it, res["streams"]["hires"], res["trajectory"]))
        else:
            p["rng"].revert()
            if use_bridges:
                for o, (a, b) in zip(obs, old):
                    o["target_shift"], o["target_slope"] = a, b
    print(f"rho={rho}, bridges={use_bridges}: {len(accepted)} of {count} proposals accepted")
    return accepted, obs

### Figure 41 (p. 106)

In [ ]:
panels = [("Uncorrelated", 0.0, False), ("Strong Correlation", 0.999, False),
          ("Medium Correlation", 0.99, False), ("Our Bridge Transformation", 0.0, True)]
fig, axes = plt.subplots(2, 2, figsize=(12, 9.5))
cmap = plt.get_cmap("winter", 3000)
def free_corner(ax, x, y, w=0.42, h=0.38):
    """Inset position [x0, y0, w, h] (axes fraction) in the corner that contains the fewest data points."""
    X, Y = ax.transLimits.transform(np.column_stack([x, y])).T
    corners = [(0, 1 - h), (1 - w, 1 - h), (0, 0), (1 - w, 0)]
    count = [np.sum((X > cx) & (X < cx + w) & (Y > cy) & (Y < cy + h)) for cx, cy in corners]
    return [*corners[int(np.argmin(count))], w, h]

for ax, (title, rho, use_br) in zip(axes.flat, panels):
    acc, obs = chain(initial_observations(4), rho, use_br)
    last = obs[-1]["Y"]
    for it, H, _ in acc:
        ax.plot(H[0], H[1], "-", lw=0.4, c=cmap(it))
    ax.autoscale_view()
    allx = np.concatenate([H[0] for _, H, _ in acc]); ally = np.concatenate([H[1] for _, H, _ in acc])
    ins = ax.inset_axes(free_corner(ax, allx, ally))
    for it, H, _ in acc:
        ins.plot(H[0], H[1], "-", lw=0.4, c=cmap(it))
    pts = np.array([(0.0, 0.0)] + [tuple(o["Y"][:2]) for o in obs])
    for a in (ax, ins):
        a.plot(pts[:, 0], pts[:, 1], "o", color="r", ms=7, zorder=10)
    ins.set(xlim=(last[0] - 0.06, last[0] + 0.01), ylim=(last[1] - 0.02, last[1] + 0.02), xticks=[], yticks=[])
    ax.set_title(title, fontsize=14)
    ax.plot([], [], "o", color="r", label="observations (start at the origin)")
    ax.plot([], [], color=cmap(0), label="accepted samples, early (blue) → late (green)")
    ax.legend(loc="lower right", fontsize=7)
fig.tight_layout()
savefig("fig41_sticky_chains");

**Figure 41 (p. 106).** Unique 2D trajectories sampled by CPMMH within 3000 iterations; the inset shows the arrival at the
last observation. Most samples lie on top of each other because proposals get rejected. Top left: default particle filter
(uncorrelated proposals). Top right and bottom left: CPMMH with strong and medium correlation of the Crank–Nicolson proposal.
All standard approaches show *sticky chains*: even small changes of the early rotational noise grow quadratically in time and
get rejected. Bottom right: CPMMH in the bridge space — shift and slope of every bridge are proposed with small random-walk
steps (i.e. strongly correlated), while the remaining noise is redrawn — produces many more unique samples.

### Figure 42 (p. 107): MCMC warm-up
A bridged chain (3000 iterations) on 6 observations; only the states at the observation times are drawn.
The bridge conditions start at zero and diffuse until the bridges have found the path.

In [ ]:
acc, obs = chain(initial_observations(6, probe=False), 0.0, True, count=3000)
cmap = plt.get_cmap("winter", 3000)
plt.figure()
for it, _, traj in acc:
    plt.plot(np.r_[0, traj[0]], np.r_[0, traj[1]], ".-", lw=0.5, ms=2, c=cmap(it), alpha=0.6)
plt.plot([o["Y"][0] for o in obs], [o["Y"][1] for o in obs], ".r", ms=6, zorder=10, label="observations")
plt.plot([], [], c=cmap(0), label="early samples (warm-up)")
plt.plot([], [], c=cmap(2999), label="late samples")
plt.legend(fontsize=8)
savefig("fig42_mcmc_warmup");

**Figure 42 (p. 107).** Initial trajectory samples of an MCMC chain, coloured from blue (early) to green (late). The early
samples form the warm-up phase, in which the sampler climbs up the tails of the posterior density.